# Data cleaning and transformation
Working file: `data/trade_raw_2019_2023.csv`, an illustrative extract with realistic quality problems. Each section matches a slide.

## Profiling the raw file

In [ ]:
import pandas as pd

raw = pd.read_csv(
    "../../data/trade_raw_2019_2023.csv")
print(raw.shape)
print(raw.dtypes["Value"])
cols = ["Reporter", "Period", "Value"]
raw[cols].sample(5, random_state=1)

## Spotting spelling variants

In [ ]:
raw["Reporter"].value_counts().tail(8)

## Work on a copy

In [ ]:
df = raw.copy()
df.columns = df.columns.str.lower()
print(list(df.columns))
print(list(raw.columns))

## The str accessor

In [ ]:
names = pd.Series([" kenya", "UK ",
                   "united states "])
print(names.str.strip().tolist())
print(names.str.strip().str.title().tolist())
print(names.str.len().tolist())

## Standardising text

In [ ]:
df["reporter"] = df["reporter"].str.strip()
print(df["reporter"].nunique(), "spellings")

## Mapping name variants

In [ ]:
NAME_MAP = {
    "USA": "United States",
    "U.S.": "United States",
    "united states": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "Vietnam": "Viet Nam",
    "germany": "Germany",
    "Federal Republic of Germany": "Germany",
}
rep = df["reporter"].replace(NAME_MAP)
df["reporter"] = rep
print(df["reporter"].nunique(), "reporters")

## to_numeric on examples

In [ ]:
examples = pd.Series(["1,234.5", "n/a",
                      "", "7400"])
text = examples.str.replace(",", "")
nums = pd.to_numeric(text, errors="coerce")
print(nums.tolist())
print(nums.isna().sum(), "missing")

## Fixing the value column

In [ ]:
value = (df["value"].astype(str)
         .str.replace(",", "", regex=False))
df["value"] = pd.to_numeric(value,
                            errors="coerce")
print(df["value"].dtype)
print(df["value"].isna().sum(), "missing")
print((df["value"] < 0).sum(), "negative")

## Updating chosen rows with loc

In [ ]:
toy = pd.DataFrame({
    "value": [5000.0, 7.4],
    "unit": ["USD thousands",
             "USD millions"]})
mask = toy["unit"] == "USD thousands"
toy.loc[mask, "value"] /= 1000
toy.loc[mask, "unit"] = "USD millions"
toy

## Standardising units

In [ ]:
thousands = df["unit"] == "USD thousands"
print(thousands.sum(), "rows in thousands")
df.loc[thousands, "value"] /= 1000
df.loc[thousands, "unit"] = "USD millions"
print(df["unit"].value_counts())

## Counting missing values

In [ ]:
print(df.isna().sum())

## Drop, fill or flag

In [ ]:
s = pd.Series([12.0, None, 30.0])
print(s.dropna().tolist())    # drop
print(s.fillna(0).tolist())   # fill
print(s.isna().tolist())      # flag
print(s.mean(), s.fillna(0).mean())

## Exact duplicates

In [ ]:
print(df.duplicated().sum(), "exact dups")
df = df.drop_duplicates()
print(len(df), "rows left")

## Date format codes

In [ ]:
d1 = pd.to_datetime("31/12/2021",
                    format="%d/%m/%Y")
d2 = pd.to_datetime("Dec 2021",
                    format="%b %Y")
print(d1, d1.year)
print(d2, d2.month)
print(pd.to_datetime("01/02/2023",
                     format="%d/%m/%Y"))

## Parsing several formats

In [ ]:
FORMATS = ["%Y-%m-%d", "%d/%m/%Y",
           "%b %Y", "%Y"]
def parse_period(p):
    for f in FORMATS:
        try:
            return pd.to_datetime(p, format=f)
        except ValueError:
            continue
    return pd.NaT
df["date"] = df["period"].map(parse_period)
df["year"] = df["date"].dt.year
df[["period", "date", "year"]].head(3)

## Key duplicates

In [ ]:
key = ["reporter", "partner", "year"]
dups = df[df.duplicated(subset=key,
                        keep=False)]
print(len(dups), "rows share a key")
dups.sort_values(key)[
    ["reporter", "period", "value"]].head(4)

## How merge matches rows

In [ ]:
left = pd.DataFrame({
    "reporter": ["Kenya", "Ghana", "Utopia"],
    "exports": [7.4, 17.3, 1.0]})
right = pd.DataFrame({
    "country_name": ["Kenya", "Ghana"],
    "iso3": ["KEN", "GHA"]})
left.merge(right, left_on="reporter",
           right_on="country_name",
           how="left")

## Merge safety checks

In [ ]:
countries = pd.read_excel(
    "../../data/countries.xlsx")
clean = df.merge(
    countries[["iso3", "country_name"]],
    left_on="reporter",
    right_on="country_name",
    how="left", validate="many_to_one",
    indicator=True)
print(clean["_merge"].value_counts())

## Stacking tables with concat

In [ ]:
y2022 = pd.DataFrame({"iso3": ["KEN"],
                      "year": [2022]})
y2023 = pd.DataFrame({"iso3": ["KEN"],
                      "year": [2023]})
both = pd.concat([y2022, y2023],
                 ignore_index=True)
both

## Merging real World Bank data

In [ ]:
wb = pd.read_csv(
    "../../data/wb_indicators_2015_2023.csv")
world = clean[clean["partner"] == "World"]
combined = world.merge(
    wb[["iso3", "year", "gdp_usd"]],
    on=["iso3", "year"], how="left")
combined["exports_pct_gdp"] = (
    combined["value"] * 1e6
    / combined["gdp_usd"] * 100).round(1)
combined[["reporter", "year",
          "exports_pct_gdp"]].head(3)

## Validation

In [ ]:
def validate(df):
    problems = []
    if df["value"].lt(0).any():
        problems.append("negative values")
    if df["iso3"].isna().any():
        problems.append("unmatched reporters")
    key = ["iso3", "partner", "year"]
    if df.duplicated(key).any():
        problems.append("duplicate keys")
    return problems

print(validate(clean) or "All checks passed")